# 010 — Labellisation offline
Règles versionnées, labels expérimentaux. Aucun appel API.

In [ ]:
from pathlib import Path
import sys,json
from collections import Counter
import pandas as pd
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"gdai-plant").is_dir())
PLANT = ROOT/"gdai-plant"
sys.path.insert(0,str(PLANT))
from matching.label_candidate_pairs import build
from enrichment.prepare_claude_review import digest
from matching.dataset_quality import audit
read = lambda p: json.loads(p.read_text(encoding="utf-8"))
GARDENS = PLANT/"data/scenarios/garden-scenarios-idf-v1.json"
PROFILE = PLANT/"config/dataset-scope-idf-v1.json"
CATALOGUE = PLANT/"data/plants-mvp-100.json"
QUALIFIED = PLANT/"data/enrichment/mvp-100/qualified-v1"
RULES = PLANT/"config/matching-rules-v3.json"
gardens_doc, profile, catalogue, config = map(read,(GARDENS,PROFILE,CATALOGUE,RULES))
check = audit(gardens_doc,profile)
assert check["flagged"] == 0 and check["target_count_met"]
profiles, quality, report = [read(QUALIFIED/n) for n in ("profiles-provisional.json","quality.json","qualification-report.json")]
assert report["catalogue_sha256"] == digest(catalogue)
assert report["processed"] == len(catalogue) and not report["missing"]
assert {p["scientific_name"] for p in profiles["profiles"]} == {p["scientific_name"] for p in catalogue}

In [ ]:
categories = {p["scientific_name"]:p["candidate_category"] for p in catalogue}
# No assumed user commitments: compensation disabled unless explicitly modelled.
rows = build(gardens_doc["scenarios"],profiles,quality,"v4",categories,config,synthetic_compensation=False)
assert len(rows) == len(gardens_doc["scenarios"])*len(catalogue)
eligible = [r for r in rows if r["experimental_training_eligible"]]
coverage = Counter(r["garden_id"] for r in eligible if r["relevance_grade"] >= 2)
summary = {"status":"EXPERIMENTAL_NOT_EXPERT_VALIDATED","dataset_profile_id":profile["profile_id"],"rule_set_id":"horticultural-rules-v4","garden_sha256":digest(gardens_doc),"catalogue_sha256":digest(catalogue),"profiles_sha256":digest(profiles),"quality_sha256":digest(quality),"rule_config_sha256":digest(config),"pairs":len(rows),"eligible":len(eligible),"eligible_label_counts":dict(Counter(str(r["relevance_grade"]) for r in eligible)),"eligible_plants":len({r["scientific_name"] for r in eligible}),"gardens_with_at_least_five":sum(v>=5 for v in coverage.values()),"compensation_mode":"NO_UNVERIFIED_COMMITMENTS","api_calls":0}
print(json.dumps(summary,indent=2))
display(pd.DataFrame(eligible)["relevance_grade"].value_counts() if eligible else pd.Series(dtype=int))

In [ ]:
assert eligible and len({r["scientific_name"] for r in eligible}) >= 5, "Insufficient eligible plants"
OUT = PLANT/"data/matching/fr-idf-v1"
OUT.mkdir(parents=True,exist_ok=False)
for name,data in [("labelled-pairs.json",rows),("matching-report.json",summary)]:
    (OUT/name).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding="utf-8")
print(OUT)